In [15]:
import os

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv(override=True)
DEEPSEEK_API_KEY= os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL= os.getenv("DEEPSEEK_BASE_URL")
# OPENAI_API_KEY= os.getenv("OPENAI_API_KEY")
# OPENAI_BASE_URL= os.getenv("OPENAI_BASE_URL")
llmOpenai = init_chat_model(
    model="deepseek:deepseek-v4-flash",
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
    # api_key=OPENAI_API_KEY,
    # base_url=OPENAI_BASE_URL,
)



In [4]:
from langchain_core.messages import HumanMessage, ToolMessage
from pydantic import BaseModel, Field
from langchain_core.tools import tool
from rich import print as rprint

class Person(BaseModel):
    """人物信息"""
    name: str = Field(description="姓名")
    age: int = Field(description="年龄")
    occupation: str = Field(description="职业")

structured_model = llmOpenai.with_structured_output(Person)

response = structured_model.invoke("你是谁")
print(response)

name='ChatGPT' age=0 occupation='AI助手'


In [5]:
from pydantic import BaseModel, Field, SecretStr
class MovieModel(BaseModel):
    """
    电影的详细信息
    """
    title: str = Field(description="电影标题")
    year: int = Field(description="电影上映年份")
    director: str = Field(description="导演")
    rating: float = Field(description="电影评分，满分十分")

model_with_structure = llmOpenai.with_structured_output(MovieModel)
response = model_with_structure.invoke("给出盗梦空间的信息")

print(response)
print(type(response))

title='盗梦空间' year=2010 director='克里斯托弗·诺兰' rating=8.8
<class '__main__.MovieModel'>


## 情况1：可选字段

In [6]:
from pydantic import BaseModel, Field
from typing import Optional
class Person(BaseModel):
    """人物信息"""
    name: str = Field(description="姓名")
    age: int = Field(description="年龄")
    occupation: str = Field(description="职业")
structured_llm = llmOpenai.with_structured_output(Person)
response = structured_llm.invoke("张三是一名医生")
print(response)

name='张三' age=0 occupation='医生'


In [7]:
from pydantic import BaseModel, Field
from typing import Optional
class Person(BaseModel):
    """人物信息"""
    name: str = Field(description="姓名")
    age: Optional[int] = Field(description="年龄")
    occupation: str = Field(description="职业")
structured_llm = llmOpenai.with_structured_output(Person)
response = structured_llm.invoke("张三是一名医生")
print(response)

name='张三' age=None occupation='医生'


## 情况2：默认值

In [21]:
from typing import Optional
from pydantic import BaseModel, Field
class Person(BaseModel):
    """人物信息"""
    name: str = Field(description="姓名")
    age: int = Field(14,description="年龄")
    occupation: str = Field(description="职业")
# ✅ 在 with_structured_output 中传递参数禁用思考模式
structured_llm = llmOpenai.with_structured_output(
    Person,
    method="function_calling",
    include_raw=False,
    # 尝试传入 extra_body 参数
    extra_body={
        "thinking": {
            "type": "disabled"  # 尝试禁用思考模式
        }
    } # 如果 DeepSeek 支持此参数
)
response = structured_llm.invoke("张三是一名医生")
print(response)

name='张三' age=14 occupation='医生'
